# FahMai RAG Challenge
### Retrieval-Augmented Generation Pipeline

---

**Author:** Super AI Engineer Season 6  
**Version:** 2.0  
**Last Updated:** March 2026

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Core Abstractions](#2-core-abstractions)
3. [Data Layer](#3-data-layer)
4. [Retrieval Engine](#4-retrieval-engine)
5. [LLM Service](#5-llm-service)
6. [Inference Pipeline](#6-inference-pipeline)
7. [Execution](#7-execution)
8. [Evaluation & Export](#8-evaluation--export)

---

## 1. Environment Setup

Install dependencies and configure the runtime environment.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.1 Dependencies
# ══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

REQUIRED_PACKAGES = [
    "rank_bm25",
    "scikit-learn",
    "pandas",
    "tqdm",
    "python-dotenv",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES]
)

print("Dependencies installed")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.2 Imports
# ══════════════════════════════════════════════════════════════════════════════

from __future__ import annotations

import json
import logging
import os
import re
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum, auto
from functools import reduce, wraps
from pathlib import Path
from typing import (
    Any,
    Callable,
    Dict,
    Generic,
    Iterator,
    List,
    Optional,
    Protocol,
    Tuple,
    TypeVar,
)

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

load_dotenv()

T = TypeVar("T")
R = TypeVar("R")

print("Imports complete")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.3 Configuration
# ══════════════════════════════════════════════════════════════════════════════

class Environment(Enum):
    DEVELOPMENT = auto()
    PRODUCTION = auto()


@dataclass(frozen=True)
class Settings:
    """Application settings."""
    
    # API
    api_key: str = field(default_factory=lambda: os.getenv("API_KEY", "zEJ2JyfKmDAguvcUgVndXkNGF7g19vsf"))
    api_timeout: int = 120
    api_max_retries: int = 5
    
    # Paths
    knowledge_base_dir: Path = field(default_factory=lambda: Path("data/knowledge_base"))
    questions_file: Path = field(default_factory=lambda: Path("data/questions.csv"))
    checkpoint_file: Path = field(default_factory=lambda: Path("checkpoint.json"))
    output_file: Path = field(default_factory=lambda: Path("submission.csv"))
    
    # Retrieval
    chunk_size: int = 500
    chunk_overlap: int = 100
    retrieval_top_k: int = 8
    
    # LLM
    max_tokens: int = 1024
    temperature: float = 0.1
    
    # Pipeline
    rate_limit_delay: float = 0.3
    verification_threshold: float = 0.7
    
    env: Environment = Environment.PRODUCTION


SETTINGS = Settings()

print(f"Settings loaded | Env: {SETTINGS.env.name}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.4 Logging
# ══════════════════════════════════════════════════════════════════════════════

class LoggerFactory:
    @staticmethod
    def create(name: str, log_file: Optional[Path] = None) -> logging.Logger:
        logger = logging.getLogger(name)
        logger.setLevel(logging.DEBUG)
        logger.handlers.clear()
        
        console = logging.StreamHandler()
        console.setLevel(logging.INFO)
        console.setFormatter(logging.Formatter("%(message)s"))
        logger.addHandler(console)
        
        if log_file:
            fh = logging.FileHandler(log_file, encoding="utf-8")
            fh.setLevel(logging.DEBUG)
            fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
            logger.addHandler(fh)
        
        return logger


LOG_FILE = Path(f"run_{datetime.now():%Y%m%d_%H%M%S}.log")
logger = LoggerFactory.create("fahmai", LOG_FILE)

print(f"Logger initialized -> {LOG_FILE}")

---

## 2. Core Abstractions

Foundational types, protocols, and utilities.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.1 Result Type
# ══════════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class Ok(Generic[T]):
    value: T
    def is_ok(self) -> bool: return True
    def unwrap(self) -> T: return self.value
    def unwrap_or(self, default: T) -> T: return self.value
    def map(self, fn: Callable[[T], R]) -> Ok[R]: return Ok(fn(self.value))


@dataclass(frozen=True)
class Err(Generic[T]):
    error: str
    def is_ok(self) -> bool: return False
    def unwrap(self) -> T: raise ValueError(self.error)
    def unwrap_or(self, default: T) -> T: return default
    def map(self, fn: Callable[[T], R]) -> Err[R]: return Err(self.error)


Result = Ok[T] | Err[T]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.2 Retry Decorator
# ══════════════════════════════════════════════════════════════════════════════

def retry(
    max_attempts: int = 3,
    delay: float = 1.0,
    backoff: float = 2.0,
    exceptions: Tuple[type, ...] = (Exception,),
) -> Callable:
    def decorator(fn: Callable[..., T]) -> Callable[..., T]:
        @wraps(fn)
        def wrapper(*args, **kwargs) -> T:
            last_error = None
            current_delay = delay
            
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    last_error = e
                    if attempt < max_attempts:
                        logger.debug(f"Retry {attempt}/{max_attempts}: {e}")
                        time.sleep(current_delay)
                        current_delay *= backoff
            
            raise last_error
        return wrapper
    return decorator

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.3 Functional Utilities
# ══════════════════════════════════════════════════════════════════════════════

pipe = lambda value, *fns: reduce(lambda acc, fn: fn(acc), fns, value)
compose = lambda *fns: lambda x: reduce(lambda acc, fn: fn(acc), reversed(fns), x)
flatten = lambda nested: [item for sublist in nested for item in sublist]

print("Core abstractions defined")

---

## 3. Data Layer

Domain models and data access.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.1 Domain Models
# ══════════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class Document:
    id: str
    category: str
    filename: str
    content: str


@dataclass(frozen=True)
class Chunk:
    id: str
    document_id: str
    text: str
    score: float = 0.0


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    choices: Tuple[str, ...]


@dataclass(frozen=True)
class Answer:
    question_id: int
    choice: int
    confidence: float
    method: str

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.2 Text Processor
# ══════════════════════════════════════════════════════════════════════════════

class TextProcessor:
    THAI_PATTERN = re.compile(r"[\u0E00-\u0E7F]+|[a-zA-Z0-9]+")
    THINK_PATTERN = re.compile(r"<think>[\s\S]*?</think>", re.IGNORECASE)
    ANSWER_PATTERNS = [
        re.compile(r"ANSWER:\s*(\d+)", re.IGNORECASE),
        re.compile(r"<answer>\s*(\d+)\s*</answer>", re.IGNORECASE),
        re.compile(r"คำตอบ[:\s]*(\d+)"),
        re.compile(r"\b(10|[1-9])\b"),
    ]
    
    @staticmethod
    def tokenize(text: str) -> List[str]:
        tokens = TextProcessor.THAI_PATTERN.findall(text.lower())
        return [t for t in tokens if len(t) >= 2]
    
    @staticmethod
    def remove_think_tags(text: str) -> str:
        return TextProcessor.THINK_PATTERN.sub("", text).strip()
    
    @staticmethod
    def extract_answer(text: str) -> Optional[int]:
        if not text:
            return None
        clean = TextProcessor.remove_think_tags(text)
        for pattern in TextProcessor.ANSWER_PATTERNS:
            if match := pattern.search(clean):
                num = int(match.group(1))
                if 1 <= num <= 10:
                    return num
        return None

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.3 Knowledge Base Repository
# ══════════════════════════════════════════════════════════════════════════════

class KnowledgeBaseRepository:
    def __init__(self, base_path: Path):
        self._base_path = base_path
        self._documents: List[Document] = []
        self._chunks: List[Chunk] = []
    
    def load(self, chunk_size: int, chunk_overlap: int) -> "KnowledgeBaseRepository":
        self._documents = self._load_documents()
        self._chunks = self._create_chunks(chunk_size, chunk_overlap)
        return self
    
    def _load_documents(self) -> List[Document]:
        docs = []
        for path in sorted(self._base_path.rglob("*.md")):
            try:
                content = path.read_text(encoding="utf-8").strip()
                rel_parts = path.relative_to(self._base_path).parts
                category = rel_parts[0] if len(rel_parts) > 1 else "root"
                docs.append(Document(path.stem, category, path.name, content))
            except Exception as e:
                logger.warning(f"Failed to read {path}: {e}")
        return docs
    
    def _create_chunks(self, chunk_size: int, overlap: int) -> List[Chunk]:
        chunks = []
        for doc in self._documents:
            content = doc.content
            if len(content) <= chunk_size:
                chunks.append(Chunk(f"{doc.id}_0", doc.id, content))
            else:
                start, idx = 0, 0
                while start < len(content):
                    end = min(start + chunk_size, len(content))
                    chunks.append(Chunk(f"{doc.id}_{idx}", doc.id, content[start:end]))
                    idx += 1
                    start += chunk_size - overlap
        return chunks
    
    @property
    def documents(self) -> List[Document]: return self._documents
    
    @property
    def chunks(self) -> List[Chunk]: return self._chunks
    
    def summary(self) -> Dict:
        from collections import Counter
        return {
            "documents": len(self._documents),
            "chunks": len(self._chunks),
            "categories": dict(Counter(d.category for d in self._documents)),
        }

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.4 Question Repository
# ══════════════════════════════════════════════════════════════════════════════

class QuestionRepository:
    def __init__(self, file_path: Path):
        self._file_path = file_path
        self._questions: List[Question] = []
    
    def load(self) -> "QuestionRepository":
        df = pd.read_csv(self._file_path)
        q_col = next((c for c in ["question", "Question"] if c in df.columns), df.columns[1])
        choice_cols = sorted([c for c in df.columns if "choice" in c.lower()], 
                            key=lambda x: int(re.search(r"\d+", x).group()))
        
        self._questions = [
            Question(int(row["id"]), str(row[q_col]), tuple(str(row[c]) for c in choice_cols))
            for _, row in df.iterrows()
        ]
        return self
    
    @property
    def questions(self) -> List[Question]: return self._questions
    def __len__(self) -> int: return len(self._questions)
    def __iter__(self) -> Iterator[Question]: return iter(self._questions)


print("Data layer defined")

---

## 4. Retrieval Engine

Hybrid retrieval combining BM25 and TF-IDF.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.1 Retriever Protocol
# ══════════════════════════════════════════════════════════════════════════════

class Retriever(Protocol):
    def retrieve(self, query: str, top_k: int) -> List[Chunk]: ...

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.2 BM25 Retriever
# ══════════════════════════════════════════════════════════════════════════════

class BM25Retriever:
    def __init__(self, chunks: List[Chunk]):
        self._chunks = chunks
        self._index = BM25Okapi([TextProcessor.tokenize(c.text) for c in chunks])
    
    def retrieve(self, query: str, top_k: int) -> List[Chunk]:
        scores = self._index.get_scores(TextProcessor.tokenize(query))
        indices = np.argsort(scores)[::-1][:top_k]
        return [
            Chunk(self._chunks[i].id, self._chunks[i].document_id, 
                  self._chunks[i].text, float(scores[i]))
            for i in indices if scores[i] > 0
        ]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.3 TF-IDF Retriever
# ══════════════════════════════════════════════════════════════════════════════

class TfidfRetriever:
    def __init__(self, chunks: List[Chunk]):
        self._chunks = chunks
        self._vectorizer = TfidfVectorizer(
            tokenizer=TextProcessor.tokenize, token_pattern=None,
            min_df=1, max_features=50000, sublinear_tf=True
        )
        self._matrix = self._vectorizer.fit_transform([c.text for c in chunks])
    
    def retrieve(self, query: str, top_k: int) -> List[Chunk]:
        q_vec = self._vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self._matrix).flatten()
        indices = np.argsort(scores)[::-1][:top_k]
        return [
            Chunk(self._chunks[i].id, self._chunks[i].document_id,
                  self._chunks[i].text, float(scores[i]))
            for i in indices if scores[i] > 0
        ]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.4 Hybrid Retriever
# ══════════════════════════════════════════════════════════════════════════════

class HybridRetriever:
    def __init__(self, chunks: List[Chunk]):
        self._retrievers = [
            (BM25Retriever(chunks), 0.5),
            (TfidfRetriever(chunks), 0.5),
        ]
    
    def retrieve(self, query: str, top_k: int) -> List[Chunk]:
        scores: Dict[str, Tuple[Chunk, float]] = {}
        
        for retriever, weight in self._retrievers:
            results = retriever.retrieve(query, top_k)
            if not results:
                continue
            max_score = max(r.score for r in results) or 1.0
            for chunk in results:
                norm = (chunk.score / max_score) * weight
                if chunk.id in scores:
                    c, s = scores[chunk.id]
                    scores[chunk.id] = (c, s + norm)
                else:
                    scores[chunk.id] = (chunk, norm)
        
        ranked = sorted(scores.values(), key=lambda x: x[1], reverse=True)
        return [Chunk(c.id, c.document_id, c.text, s) for c, s in ranked[:top_k]]


print("Retrieval engine defined")

---

## 5. LLM Service

Language model client with retry and response parsing.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.1 Model Configuration
# ══════════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class ModelConfig:
    name: str
    endpoint: str
    weight: float = 1.0


MODELS = (
    ModelConfig("typhoon", "http://thaillm.or.th/api/typhoon/v1", 1.5),
    ModelConfig("openthaigpt", "http://thaillm.or.th/api/openthaigpt/v1", 1.3),
    ModelConfig("kbtg", "http://thaillm.or.th/api/kbtg/v1", 1.0),
    ModelConfig("pathumma", "http://thaillm.or.th/api/pathumma/v1", 1.0),
)


@dataclass(frozen=True)
class LLMResponse:
    model: str
    raw_text: Optional[str]
    answer: Optional[int]
    success: bool

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.2 LLM Client
# ══════════════════════════════════════════════════════════════════════════════

class LLMClient:
    def __init__(self, settings: Settings):
        self._settings = settings
    
    @retry(max_attempts=5, delay=2.0, backoff=2.0, exceptions=(requests.RequestException, KeyError))
    def _call_api(self, model: ModelConfig, messages: List[Dict]) -> str:
        resp = requests.post(
            f"{model.endpoint}/chat/completions",
            headers={"Content-Type": "application/json", "apikey": self._settings.api_key},
            json={"model": "/model", "messages": messages,
                  "max_tokens": self._settings.max_tokens,
                  "temperature": self._settings.temperature},
            timeout=self._settings.api_timeout,
        )
        if resp.status_code == 429:
            raise requests.RequestException("Rate limited")
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"]
    
    def complete(self, model: ModelConfig, messages: List[Dict]) -> LLMResponse:
        try:
            raw = self._call_api(model, messages)
            clean = TextProcessor.remove_think_tags(raw)
            answer = TextProcessor.extract_answer(clean)
            return LLMResponse(model.name, clean, answer, True)
        except Exception as e:
            logger.debug(f"LLM error ({model.name}): {e}")
            return LLMResponse(model.name, None, None, False)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.3 Prompt Builder
# ══════════════════════════════════════════════════════════════════════════════

class PromptBuilder:
    SYSTEM = """คุณเป็น AI ร้านฟ้าใหม่ ตอบตามฐานข้อมูลเท่านั้น
กฎ: 1) ตรวจสอบทุกตัวเลือก 2) ต้องถูก 100% 3) ผิดนิดเดียว=ผิดหมด
4) ไม่มีข้อถูก→9 5) ไม่เกี่ยว→10 ตอบตัวเลขเท่านั้น"""
    
    @staticmethod
    def build(question: Question, context: List[Chunk]) -> List[Dict]:
        ctx = "\n".join(f"[{c.document_id}] {c.text}" for c in context) or "ไม่พบข้อมูล"
        choices = "\n".join(f"{i+1}. {c}" for i, c in enumerate(question.choices))
        return [
            {"role": "system", "content": PromptBuilder.SYSTEM},
            {"role": "user", "content": f"ข้อมูล:\n{ctx}\n\nคำถาม: {question.text}\n\nตัวเลือก:\n{choices}\n\nตอบ:"},
        ]


print("LLM service defined")

---

## 6. Inference Pipeline

Orchestrate retrieval, inference, and verification.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.1 Voting Strategy
# ══════════════════════════════════════════════════════════════════════════════

class VotingStrategy:
    @staticmethod
    def weighted_vote(responses: List[LLMResponse], weights: Dict[str, float]) -> Tuple[int, float]:
        votes: Dict[int, float] = {}
        for r in responses:
            if r.answer:
                votes[r.answer] = votes.get(r.answer, 0) + weights.get(r.model, 1.0)
        if not votes:
            return (9, 0.0)
        total = sum(votes.values())
        winner = max(votes, key=votes.get)
        return (winner, votes[winner] / total)
    
    @staticmethod
    def check_consensus(responses: List[LLMResponse]) -> Optional[int]:
        answers = [r.answer for r in responses if r.answer]
        return answers[0] if len(answers) >= 2 and len(set(answers)) == 1 else None

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.2 Question Classifier
# ══════════════════════════════════════════════════════════════════════════════

class QuestionClassifier:
    UNRELATED = frozenset(["วันหยุด", "ตั๋วเครื่องบิน", "ดอกเบี้ย", "สูตรอาหาร", "โควิด", "หุ้น", "bitcoin"])
    
    @classmethod
    def is_unrelated(cls, q: Question) -> bool:
        return any(kw in q.text.lower() for kw in cls.UNRELATED)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.3 Inference Engine
# ══════════════════════════════════════════════════════════════════════════════

class InferenceEngine:
    def __init__(self, retriever: HybridRetriever, llm: LLMClient,
                 models: Tuple[ModelConfig, ...], settings: Settings):
        self._retriever = retriever
        self._llm = llm
        self._models = models[:3]
        self._settings = settings
        self._weights = {m.name: m.weight for m in models}
    
    def answer(self, question: Question) -> Answer:
        # Quick classify
        if QuestionClassifier.is_unrelated(question):
            logger.info(f"  Q{question.id}: Unrelated -> 10")
            return Answer(question.id, 10, 1.0, "classifier")
        
        # Retrieve
        query = question.text + " " + " ".join(question.choices[:5])
        context = self._retriever.retrieve(query, self._settings.retrieval_top_k)
        messages = PromptBuilder.build(question, context)
        
        # Get responses
        responses = []
        for m in self._models:
            print(f"  {m.name}...", end=" ", flush=True)
            r = self._llm.complete(m, messages)
            print(f"-> {r.answer}")
            responses.append(r)
        
        # Consensus check
        if consensus := VotingStrategy.check_consensus(responses):
            logger.info(f"  Q{question.id}: Consensus -> {consensus}")
            return Answer(question.id, consensus, 1.0, "consensus")
        
        # Weighted vote
        answer, conf = VotingStrategy.weighted_vote(responses, self._weights)
        
        # Verify if low confidence
        if conf < self._settings.verification_threshold:
            answer, conf = self._verify(question, context, answer, responses)
        
        logger.info(f"  Q{question.id}: Vote -> {answer} (conf={conf:.2f})")
        return Answer(question.id, answer, conf, "vote")
    
    def _verify(self, q: Question, ctx: List[Chunk], proposed: int, 
                responses: List[LLMResponse]) -> Tuple[int, float]:
        print(f"  Verify {proposed}...", end=" ", flush=True)
        msg = [{"role": "user", "content": f"ตรวจ: คำตอบ {proposed} ถูกไหม? คำถาม: {q.text}"}]
        r = self._llm.complete(self._models[0], msg)
        print(f"-> {r.answer}")
        
        if r.answer and r.answer != proposed:
            all_r = responses + [r]
            weights = {**self._weights, r.model: 2.0}
            return VotingStrategy.weighted_vote(all_r, weights)
        return (proposed, 0.5)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.4 Checkpoint Manager
# ══════════════════════════════════════════════════════════════════════════════

class CheckpointManager:
    def __init__(self, path: Path):
        self._path = path
    
    def load(self) -> Dict[int, Answer]:
        if not self._path.exists():
            return {}
        try:
            data = json.loads(self._path.read_text(encoding="utf-8"))
            return {int(k): Answer(int(k), v["answer"], v.get("confidence", 1.0), 
                                   v.get("method", "ckpt")) for k, v in data.items()}
        except Exception:
            return {}
    
    def save(self, answers: Dict[int, Answer]) -> None:
        data = {str(k): {"answer": a.choice, "confidence": a.confidence, 
                         "method": a.method} for k, a in answers.items()}
        self._path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.5 Pipeline
# ══════════════════════════════════════════════════════════════════════════════

class Pipeline:
    def __init__(self, engine: InferenceEngine, checkpoint: CheckpointManager, settings: Settings):
        self._engine = engine
        self._checkpoint = checkpoint
        self._settings = settings
    
    def run(self, questions: List[Question], resume: bool = True) -> Dict[int, Answer]:
        answers = self._checkpoint.load() if resume else {}
        remaining = [q for q in questions if q.id not in answers]
        
        logger.info(f"Pipeline: {len(answers)} done, {len(remaining)} remaining")
        
        for q in tqdm(remaining, desc="Processing"):
            print(f"\nQ{q.id}: {q.text[:50]}...")
            answers[q.id] = self._engine.answer(q)
            self._checkpoint.save(answers)
            time.sleep(self._settings.rate_limit_delay)
        
        logger.info(f"Pipeline complete: {len(answers)} answers")
        return answers


print("Inference pipeline defined")

---

## 7. Execution

Initialize and run the pipeline.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.1 Initialize
# ══════════════════════════════════════════════════════════════════════════════

kb_repo = KnowledgeBaseRepository(SETTINGS.knowledge_base_dir).load(
    SETTINGS.chunk_size, SETTINGS.chunk_overlap
)
print(f"KB: {kb_repo.summary()}")

q_repo = QuestionRepository(SETTINGS.questions_file).load()
print(f"Questions: {len(q_repo)}")

retriever = HybridRetriever(kb_repo.chunks)
llm = LLMClient(SETTINGS)
engine = InferenceEngine(retriever, llm, MODELS, SETTINGS)
checkpoint = CheckpointManager(SETTINGS.checkpoint_file)
pipeline = Pipeline(engine, checkpoint, SETTINGS)

print("\nAll components initialized")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.2 Run
# ══════════════════════════════════════════════════════════════════════════════

answers = pipeline.run(q_repo.questions, resume=True)

---

## 8. Evaluation & Export

Generate submission and analyze results.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.1 Export
# ══════════════════════════════════════════════════════════════════════════════

submission = pd.DataFrame([{"id": k, "answer": v.choice} for k, v in sorted(answers.items())])
submission.to_csv(SETTINGS.output_file, index=False)

print(f"Exported: {SETTINGS.output_file} ({len(submission)} rows)")
submission.head(10)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.2 Validate
# ══════════════════════════════════════════════════════════════════════════════

checks = [
    (list(submission.columns) == ["id", "answer"], "Columns OK"),
    (len(submission) == len(q_repo), "Row count OK"),
    (submission["answer"].between(1, 10).all(), "Values OK"),
    (not submission["id"].duplicated().any(), "No duplicates"),
]

for passed, msg in checks:
    print(f"{'[OK]' if passed else '[FAIL]'} {msg}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.3 Analysis
# ══════════════════════════════════════════════════════════════════════════════

print("Answer Distribution:")
dist = submission["answer"].value_counts().sort_index()
for ans, cnt in dist.items():
    pct = cnt / len(submission) * 100
    label = " (no data)" if ans == 9 else " (unrelated)" if ans == 10 else ""
    print(f"  {ans:2d}: {cnt:3d} ({pct:5.1f}%){label}")

print(f"\nConfidence: mean={np.mean([a.confidence for a in answers.values()]):.2f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.4 Debug
# ══════════════════════════════════════════════════════════════════════════════

def debug(qid: int):
    q = next((x for x in q_repo.questions if x.id == qid), None)
    if not q: return print(f"Q{qid} not found")
    
    print(f"Q{qid}: {q.text}")
    for i, c in enumerate(q.choices, 1): print(f"  {i}. {c}")
    
    ctx = retriever.retrieve(q.text, 3)
    print(f"\nContext:")
    for c in ctx: print(f"  [{c.document_id}] {c.text[:80]}...")
    
    if qid in answers:
        a = answers[qid]
        print(f"\nAnswer: {a.choice} ({a.method}, conf={a.confidence:.2f})")

# debug(1)